## Accesing API Key 

Install and import anthropic library

In [1]:
#!pip install anthropic
from anthropic import HUMAN_PROMPT, AI_PROMPT
import anthropic
import os
from dotenv import load_dotenv

Load the env file and get API key running

In [2]:
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

class LLM():
    def __init__(self, api_key = api_key, model = "claude-3-5-sonnet-20240620"):
        self.client = anthropic.Anthropic(api_key = api_key)
        self.model = model

    def generate(self, prompt, max_tokens=1024):
        response = self.client.messages.create(
        model=self.model,
        max_tokens=max_tokens,
        messages=[
            {"role": "user", "content": prompt}
            ]
        )
        # Extract only the response, no other info 
        return response.content[0].text.strip()

testing the api 

In [3]:
claude_example = LLM()
print(claude_example.generate("Hello, Claude"))

Hello! It's nice to meet you. How can I assist you today?


### Understanding Tone Identification 
**Tone Identification**  involves determining the tone of a given piece of text. This code will start by using sample text to evaluate how Anthropic's Claude identifies tone and compare its predictions to the ground truth to calculate an accuracy score. The accuracy score will then be compared to results from other evaluation frameworks that use packages designed to assess tone, allowing us to gauge how well each framework performs.

 **Using Sample Text Data:**

create a mini file with sample data - this is preliminary to test how claude works and get a better understanding of the processes without using a large dataset: 

In [4]:
#import pandas 
import pandas as pd

In [5]:
#create sample data 
data = {
    "Text": [
        "I am so happy to see you!",
        "This is the worst day ever!",
        "I'm feeling okay",
        "You make me so upset when you do that.",
        "What a wonderful surprise!",
        "I'm disappointed with the results.",
        "Everything is going just as planned!",
        "Lets do this next instead.",
        "This is absolutely fantastic!",
        "I feel okay about this topic."
    ],
    "Label": [
        "happy",
        "angry",
        "neutral",
        "angry",
        "happy",
        "sad",
        "happy",
        "neutral",
        "happy",
        "neutral"
    ]
}

# Convert the data into a DataFrame and only get text 
sample_data = pd.DataFrame(data)

now using this sample data we will feed it to claude to see what responses for tone it generates: 

In [6]:
# Define a function to query the API for tone analysis
def get_tone_claude(text):
    prompt = (
        f"What is the tone of this text? '{text}'\n"
        "Please choose only one tone from the following options: happy, angry, sad, or neutral."
        "Respond only with the tone (e.g., 'happy')."
    )
    return claude_example.generate(prompt)

In [7]:
#Apply the function to the dataset and store the results
sample_data["Predicted_Tone"] = sample_data["Text"].apply(get_tone_claude)

In [8]:
#view sample data
sample_data

,Text,Label,Predicted_Tone
0,I am so happy to see you!,happy,happy
1,This is the worst day ever!,angry,angry
2,I'm feeling okay,neutral,neutral
3,You make me so upset when you do that.,angry,angry
4,What a wonderful surprise!,happy,Happy
5,I'm disappointed with the results.,sad,sad
6,Everything is going just as planned!,happy,Happy
7,Lets do this next instead.,neutral,neutral
8,This is absolutely fantastic!,happy,happy
9,I feel okay about this topic.,neutral,neutral


# Assesing Tone Identification Using IBM Natural Language Understanding 
## Step 1: Prepare Your Dataset
There was no online pre-made dataset containing all the tones that IBM NL Understanding Tool has, which are: Excited, Frustrated, Impolite, Polite, Sad, Satisfied, and Sympathetic. Using OpenAI's chatgpt and human annotation, a 100 row text dataset with ground truth tones was manually created. 

In [9]:
tone_dataset = pd.read_csv("/Users/dishatrivedi/Library/CloudStorage/OneDrive-Personal/UVA Graduate College/Capstone/Tone Identification - Generated Dataset.csv")
tone_dataset.head()

,Text,Overall Tone,Specific Tone
0,I can't believe how amazing this concert is!,Positive,Excited
1,"Ugh, the traffic today is unbearable.",Negative,Frustrated
2,Could you be any more incompetent?,Negative,Impolite
3,Thank you so much for your assistance.,Positive,Polite
4,Hearing about her loss breaks my heart.,Negative,Sad


## Step 2: Set Up Environment 
IBM NL Understanding requires to get an api key to access, the watson package also comes with many different features so its important to identify which one we are using. 

In [10]:
#install IBM Watson Python SDK 
!pip install ibm-watson

In [11]:
#initialize NLU client 
from ibm_watson import NaturalLanguageUnderstandingV1
from ibm_watson.natural_language_understanding_v1 import Features, ClassificationsOptions
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator

In [12]:
#api key and service url 
load_dotenv()
IBM_key = os.getenv("IBM_API_KEY")
service_url = "https://api.us-east.natural-language-understanding.watson.cloud.ibm.com/instances/2483e3c8-25ee-478f-a85b-1d177d19e0fe"

In [13]:
#set up NLU service 
authenticator = IAMAuthenticator(IBM_key)
nlu = NaturalLanguageUnderstandingV1(
    version='2021-08-01',
    authenticator=authenticator
)
nlu.set_service_url(service_url)

## Step 3: Analyze Tone 
Use tone classification component of watson nlp to analyze tone. Previously there were errors with completion due to rate limits, so creating exception blocks to handle that among other most frequent erros encountered. 

In [14]:
#define function to analyze tone
import time
import random
import requests

def analyze_tone(text, retries=3):
    """
    Sends text to IBM Watson's NLU API for tone analysis.
    Handles rate limits (429 errors) and unsupported language errors (400).
    """
    if not text.strip():
        return "None", 1.0  # Default for empty text

    for attempt in range(retries):
        try:
            # Send request to Watson
            response = nlu.analyze(
                text=text,
                features=Features(classifications=ClassificationsOptions(model='tone-classifications-en-v1'))
            ).get_result()

            # Extract top classification
            tones = response.get("classifications", [])
            if tones:
                top_tone = max(tones, key=lambda x: x["confidence"])
                return top_tone["class_name"], top_tone["confidence"]
            
            return "None", 1.0  # Default if no tone detected

#Exceptions for runtime, language detection, and any other erros 
        except requests.exceptions.RequestException as e:
            print(f"Network error analyzing text: {text}\nError: {e}")
            return None, None  # Skip this entry

        except Exception as e:
            error_message = str(e)

            if "Too Many Requests" in error_message or "429" in error_message:
                wait_time = random.uniform(5, 15)  # Random delay to avoid triggering limits
                print(f"Rate limit hit. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)  # Pause before retrying
                continue  # Retry request

            elif "unsupported text language" in error_message or "400" in error_message:
                print(f"Unsupported language detected for text: {text}. Skipping...")
                return "Neutral", 1.0  # Default for unsupported text

            else:
                print(f"Unexpected error analyzing text: {text}\nError: {error_message}")
                return None, None  # Skip this entry

    print(f"Failed to process text after {retries} attempts: {text}")
    return None, None  # If all retries fail, return None


## Step 5: Analyze Findings
With some simple pandas manipulations I was able to do some exploratory analysis on my findings. 

In [15]:
#apply function to dataframe 
tone_dataset[['Watson_Predicted_Tone', 'Confidence']] = tone_dataset['Text'].apply(lambda x: pd.Series(analyze_tone(x)))

#briefly view updated dataset 
tone_dataset.head()

Unsupported language detected for text: You're such a nuisance.. Skipping...
Rate limit hit. Retrying in 5.6 seconds...


,Text,Overall Tone,Specific Tone,Watson_Predicted_Tone,Confidence
0,I can't believe how amazing this concert is!,Positive,Excited,excited,0.732796
1,"Ugh, the traffic today is unbearable.",Negative,Frustrated,sad,0.784093
2,Could you be any more incompetent?,Negative,Impolite,sad,0.637881
3,Thank you so much for your assistance.,Positive,Polite,polite,0.311965
4,Hearing about her loss breaks my heart.,Negative,Sad,sad,0.937166


In [16]:
#filtering out predictions with less than .5 confidence score 
#sort in desc order
tone_dataset[tone_dataset['Confidence'] >= 0.5].sort_values(by = 'Confidence', 
                                                            ascending = False)

,Text,Overall Tone,Specific Tone,Watson_Predicted_Tone,Confidence
44,You're such a nuisance.,Negative,Impolite,Neutral,1.000000
18,Losing my pet has left me heartbroken.,Negative,Sad,sad,0.970763
54,Achieving my goals gives me immense joy.,Positive,Excited,excited,0.958574
46,The gloomy weather makes me feel so depressed.,Negative,Sad,sad,0.955281
4,Hearing about her loss breaks my heart.,Negative,Sad,sad,0.937166
33,"After a long day, a warm bath makes me feel ni...",Neutral,Satisfied,excited,0.934962
66,"Ew, you are so disgusting.",Negative,Impolite,frustrated,0.909648
68,"She sat in silence, heartbroken after hearing ...",Negative,Sad,sad,0.890391
53,The loss of her childhood home made her grieve...,Negative,Sad,sad,0.879863
71,I cheered as the team scored the winning goal.,Positive,Excited,excited,0.876376


In [17]:
#filter out those that are correct 
correct = tone_dataset[tone_dataset["Specific Tone"].str.lower() == tone_dataset['Watson_Predicted_Tone'].str.lower()]

#groupby tone to see number predicted and average confidence of prediction 
correct.groupby('Specific Tone').agg({'Watson_Predicted_Tone' : 'count', 'Confidence' : 'mean'} )

,Watson_Predicted_Tone,Confidence
Specific Tone,,
Excited,9,0.700157
Frustrated,5,0.680899
Polite,12,0.436505
Sad,10,0.767532
Satisfied,3,0.563034
Sympathetic,1,0.573922


## Using VADER
Using VADER to see if indicating positive or negative tones increases accuracy of detection. 

## Step 1: Setup

In [18]:
!pip install vaderSentiment


In [19]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

## Step 2: Using the Analzer

In [20]:
def get_vader_sentiment(text):
    """Analyze sentiment using VADER and return both compound score and sentiment classification."""
    sentiment_scores = analyzer.polarity_scores(text)
    compound_score = sentiment_scores["compound"]

    # Classify based on compound score
    if compound_score >= 0.05:
        sentiment_label = "Positive"
    elif compound_score <= -0.05:
        sentiment_label = "Negative"
    else:
        sentiment_label = "Neutral"

    return compound_score, sentiment_label

# Apply VADER analysis only to the "Text" column
tone_dataset[["VADER_Compound_Score", "VADER_Predicted"]] = tone_dataset["Text"].apply(get_vader_sentiment).apply(pd.Series)

In [21]:
tone_dataset[["Text", "Overall Tone", "Specific Tone", "VADER_Predicted", "VADER_Compound_Score"]].head()

,Text,Overall Tone,Specific Tone,VADER_Predicted,VADER_Compound_Score
0,I can't believe how amazing this concert is!,Positive,Excited,Negative,-0.5210
1,"Ugh, the traffic today is unbearable.",Negative,Frustrated,Negative,-0.4215
2,Could you be any more incompetent?,Negative,Impolite,Negative,-0.5256
3,Thank you so much for your assistance.,Positive,Polite,Positive,0.3612
4,Hearing about her loss breaks my heart.,Negative,Sad,Positive,0.4404


In [22]:
#filter out those that are correct 
correct_VADER = tone_dataset[tone_dataset["Overall Tone"] == tone_dataset['VADER_Predicted']]

#groupby tone to see number predicted and average confidence of prediction 
correct_VADER.groupby('Overall Tone').agg({'VADER_Predicted' : 'count', 'VADER_Compound_Score' : 'mean'} )

,VADER_Predicted,VADER_Compound_Score
Overall Tone,,
Negative,28,-0.523207
Neutral,3,0.000000
Positive,28,0.550314


Overall Accuracy is 59%

## Brief Overview
Vader had higher accuracy compared to ibm watson tone analyzer. However it still was very limited in predicting neutrality. This may be a limit of the dataset...

# Anthropic LLM Performance - LLM as Judge 

In [ ]:
# Define a function to query the API for tone analysis - intricate tone 
def get_tone_claude(text):
    prompt = (
        f"What is the tone of this text? '{text}'\n"
        "Please choose only one tone from the following options: excited, frustrated, sympathetic, polite, impolite, satisfied, or sad."
        "Respond only with the tone (e.g., 'satisfied')."
    )
    return claude_example.generate(prompt)

In [32]:
#Apply the function to the dataset and store the results
tone_dataset["LLM Detailed Tone"] = tone_dataset["Text"].apply(get_tone_claude)

In [34]:
tone_dataset[['Text', 'Specific Tone', 'LLM Detailed Tone']]

,Text,Specific Tone,LLM Detailed Tone
0,I can't believe how amazing this concert is!,Excited,excited
1,"Ugh, the traffic today is unbearable.",Frustrated,frustrated
2,Could you be any more incompetent?,Impolite,Frustrated
3,Thank you so much for your assistance.,Polite,polite
4,Hearing about her loss breaks my heart.,Sad,Sympathetic
...,...,...,...
94,The weight of the situation was evident on eve...,Sad,Sad
95,"Every detail has been considered, and I couldn...",Satisfied,satisfied
96,Feel free to reach out if you have any other q...,Polite,Polite
97,Why do I always have to be the one to fix thes...,Frustrated,frustrated


In [35]:
#filter dataset to count accuracy 
correct_Claude = tone_dataset[tone_dataset["Specific Tone"].str.lower() == tone_dataset["LLM Detailed Tone"].str.lower()]

# Group by tone type to count correct predictions and get insights
claude_results = correct_Claude.groupby("Specific Tone").agg({
    "LLM Detailed Tone": "count"
}).rename(columns={"LLM Detailed Tone": "Correct Predictions"})

In [36]:
#view results 
claude_results

,Correct Predictions
Specific Tone,
Excited,14
Frustrated,14
Impolite,7
Polite,18
Sad,12
Satisfied,14
Sympathetic,9


Claude was 88% accurate with its predictions which is a signficant jump compared to the other two tools utilized! 

In [37]:
# Define a function to query the API for tone analysis - simple tone 
def get_tone_claude(text):
    prompt = (
        f"What is the tone of this text? '{text}'\n"
        "Please choose only one tone from the following options: negative, neutral, or positive."
        "Respond only with the tone (e.g., 'neutral')."
    )
    return claude_example.generate(prompt)

In [38]:
#Apply the function to the dataset and store the results
tone_dataset["LLM Basic Tone"] = tone_dataset["Text"].apply(get_tone_claude)

In [39]:
tone_dataset[['Text', 'Overall Tone', 'LLM Basic Tone']]

,Text,Overall Tone,LLM Basic Tone
0,I can't believe how amazing this concert is!,Positive,positive
1,"Ugh, the traffic today is unbearable.",Negative,negative
2,Could you be any more incompetent?,Negative,negative
3,Thank you so much for your assistance.,Positive,Positive
4,Hearing about her loss breaks my heart.,Negative,negative
...,...,...,...
94,The weight of the situation was evident on eve...,Negative,negative
95,"Every detail has been considered, and I couldn...",Positive,Positive
96,Feel free to reach out if you have any other q...,Neutral,Positive
97,Why do I always have to be the one to fix thes...,Negative,negative


In [42]:
#filter dataset to count accuracy 
correct_Claude2 = tone_dataset[tone_dataset["Overall Tone"].str.lower() == tone_dataset["LLM Basic Tone"].str.lower()]

# Group by tone type to count correct predictions and get insights
claude_results2 = correct_Claude2.groupby("Overall Tone").agg({
    "LLM Basic Tone": "count"
}).rename(columns={"LLM Basic Tone": "Correct Predictions"})

In [43]:
claude_results2

,Correct Predictions
Overall Tone,
Negative,38
Neutral,7
Positive,36


Claude is 81% accurate